# GPT-5.4-mini(API)를 활용한 뉴스 행위자 분석

**Actor Analysis: 개인 레벨 vs 제도 레벨 행위자 식별**

---

## 실행 환경 요약

- 모델: `gpt-5.4-mini` / API: OpenAI **Responses API** (`client.responses.create`) + JSON Schema 구조화 출력(`strict: true`)
- `reasoning_effort="none"` (분류·추출·요약 과제에 적합). `temperature`는 `reasoning_effort="none"`일 때만 지정할 수 있습니다.
- 이전 버전: [GPT-4.1 버전 저장소](https://github.com/HanJunLeeKR/News_Content_Analysis_with_GPT-4.1) (Chat Completions + `json_object`)
- 두 버전의 차이는 저장소 `README.md`의 「GPT-4.1 버전과의 차이」를 참고하세요.
---

**주요 기능:**
- ✅ 개인 레벨 행위자 추출 (정치인, 전문가, 관계자 등)
- ✅ 제도 레벨 행위자 추출 (정당, 정부기관, 단체 등)
- ✅ 쉼표로 구분된 문자열 출력
- ✅ 후속 분석(빈도, 네트워크) 용이

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Colab/Jupyter용). 로컬 환경에서는 저장소의 requirements.txt 사용:
#   pip install -r requirements.txt
# ═══════════════════════════════════════════════════════════════════════
%pip install --upgrade openai pandas tqdm nest-asyncio


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 행위자 분석기 (Actor Analysis)
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Optional, Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 행위자 식별 기준 정의
# ═══════════════════════════════════════════════════════════════════════
ANALYSIS_CRITERIA = """
**행위자(Actor) 식별 기준:**

이 분석은 뉴스 기사에 등장하는 행위자를 두 가지 레벨로 분류합니다.

**1. 개인 레벨 행위자 (Individual Actors):**
실명이 명시된 개인을 모두 포함합니다.

포함 대상:
- 정치인: "정청래", "김병기", "박수현", "이재명", "윤석열" 등
- 정부 관료: "장관", "차관", "국장" 등 (실명이 있는 경우)
- 전문가: "교수", "변호사", "의사" 등 (실명이 있는 경우)
- 기타: 기사에 실명으로 등장하는 모든 개인

제외 대상:
- 익명 정보원: "민주당 관계자", "정부 고위관계자", "A씨" 등
- 불특정 다수: "시민들", "국민", "유권자" 등

**2. 제도 레벨 행위자 (Institutional Actors):**
조직, 기관, 단체를 모두 포함합니다.

포함 대상:
- 정당: "더불어민주당", "국민의힘", "정의당" 등
- 정부기관: "대통령실", "총리실", "국회", "법무부", "검찰청" 등
- 사법기관: "헌법재판소", "대법원", "서울중앙지법" 등
- 단체/협회: "경총", "민노총", "변호사협회" 등
- 언론사: "조선일보", "한겨레" 등 (다른 언론사가 언급될 경우)
- 기업: "삼성", "현대" 등 (기업이 행위자로 등장할 경우)

제외 대상:
- 추상적 개념: "여론", "민심", "정치권" 등

**중요 원칙:**
1. 기사에 명시적으로 등장하는 행위자만 포함
2. 동일 행위자의 중복 제거 (예: "정청래", "민주당 정청래" → "정청래"만)
3. 쉼표(,)로 구분하여 나열
4. 등장 순서는 중요하지 않음
5. 행위자가 없으면 빈 문자열("") 반환
"""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """Analyze the actors appearing in the following Korean news article.

[NEWS ARTICLE]:
{text}

{criteria}

Instructions:
1. Read the article carefully and identify all actors present.
2. individual_actors: extract only named individuals (no anonymous sources).
3. institutional_actors: extract organizations, institutions, parties, and associations.
4. List each level separated by commas.
5. Remove duplicate references to the same person or institution.
6. Return an empty string if no actors exist at that level.

Output ONLY valid JSON. Do not add fields not in the schema."""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        """강화된 JSON 추출 로직"""
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")

        response_text = response_text.strip()

        # 1단계: 직접 JSON 파싱 시도
        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass

        # 2단계: 마크다운 JSON 블록 추출
        json_patterns = [
            r'```json\s*\n?(.*?)\n?```',
            r'```\s*\n?(.*?)\n?```',
            r'`{3,}\s*json\s*\n?(.*?)\n?`{3,}',
        ]

        for pattern in json_patterns:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                json_str = match.group(1).strip()
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue

        # 3단계: 중괄호 블록 추출
        brace_patterns = [
            r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',
            r'\{(?:[^{}]|{[^{}]*})*\}',
        ]

        for pattern in brace_patterns:
            matches = re.findall(pattern, response_text)
            for match in matches:
                try:
                    result = json.loads(match)
                    if EnhancedJSONParser._validate_result(result):
                        return result
                except json.JSONDecodeError:
                    continue

        # 4단계: 키-값 개별 추출
        extracted_data = {}

        individual_patterns = [
            r'"individual_actors"\s*:\s*"([^"]*(?:\\.[^"]*)*)"',
            r'individual_actors["\']?\s*[:=]\s*["\']([^"\']*)["\']',
        ]
        for pattern in individual_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE | re.DOTALL)
            if match:
                extracted_data['individual_actors'] = match.group(1).strip()
                break

        institutional_patterns = [
            r'"institutional_actors"\s*:\s*"([^"]*(?:\\.[^"]*)*)"',
            r'institutional_actors["\']?\s*[:=]\s*["\']([^"\']*)["\']',
        ]
        for pattern in institutional_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE | re.DOTALL)
            if match:
                extracted_data['institutional_actors'] = match.group(1).strip()
                break

        if extracted_data and EnhancedJSONParser._validate_result(extracted_data):
            return extracted_data

        return EnhancedJSONParser._get_default_result("파싱 실패")

    @staticmethod
    def _validate_result(result: Dict) -> bool:
        return (
            isinstance(result, dict) and
            ('individual_actors' in result or 'institutional_actors' in result)
        )

    @staticmethod
    def _get_default_result(error_msg: str) -> Dict:
        return {
            'individual_actors': '',
            'institutional_actors': '',
            'error': error_msg
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 행위자 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════
class ActorAnalyzer:
    """뉴스 행위자 분석기 (GPT-5.4-mini용)"""

    def __init__(self, api_key, model, output_dir='.',
                 concurrent_requests=2, rate_limit_delay=1.0, batch_size=5,
                 reasoning_effort='none', text_verbosity='medium',
                 temperature: Optional[float] = None,
                 debug=False):
        # ── GPT-5.4-mini: temperature는 reasoning_effort='none'일 때만 유효 ──
        if temperature is not None and reasoning_effort != 'none':
            raise ValueError(
                f"temperature is only valid when reasoning_effort='none'. "
                f"Got reasoning_effort='{reasoning_effort}'. "
                f"Set reasoning_effort='none' to use temperature."
            )
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.reasoning_effort = reasoning_effort
        self.text_verbosity = text_verbosity
        self.temperature = temperature
        self.debug = debug
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)

        self.parsing_successes = 0
        self.parsing_failures = 0

        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)

        os.makedirs(self.output_dir, exist_ok=True)

    def combine_title_text(self, title: str, text: str) -> str:
        if pd.isna(title): title = ""
        if pd.isna(text): text = ""
        title = str(title).strip()
        text = str(text).strip()
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:10000]

    async def analyze_single_article(self, title: str, text: str) -> Tuple[str, str, float]:
        async with self.semaphore:
            try:
                combined_text = self.combine_title_text(title, text)
                if not combined_text:
                    return "", "", 0.0

                prompt = PROMPT_TEMPLATE.format(
                    text=combined_text,
                    criteria=ANALYSIS_CRITERIA
                )

                # ═══════════════════════════════════════════════════
                # GPT-5.4-mini API 호출
                # reasoning_effort options: none | low | medium | high | xhigh
                # temperature: only valid when reasoning_effort='none'
                # ═══════════════════════════════════════════════════
                api_params = dict(
                    model=self.model,
                    input=[{"role": "user", "content": prompt}],
                    max_output_tokens=1024,
                    reasoning={"effort": self.reasoning_effort},
                    text={
                        "verbosity": self.text_verbosity,
                        "format": {
                            "type": "json_schema",
                            "name": "actor_analysis",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "individual_actors": {
                                        "type": "string",
                                        "description": "Named individuals (comma-separated). Empty string if none."
                                    },
                                    "institutional_actors": {
                                        "type": "string",
                                        "description": "Organizations/institutions (comma-separated). Empty string if none."
                                    }
                                },
                                "required": ["individual_actors", "institutional_actors"],
                                "additionalProperties": False
                            }
                        }
                    }
                )
                # temperature는 reasoning_effort='none'일 때만 추가
                if self.temperature is not None:
                    api_params["temperature"] = self.temperature

                response = await self.client.responses.create(**api_params)

                # 응답 추출
                if hasattr(response, 'output_text') and response.output_text:
                    response_text = response.output_text
                else:
                    response_text = ""
                    for item in response.output:
                        if item.type == "message":
                            for content in item.content:
                                if hasattr(content, 'text'):
                                    response_text += content.text

                response_text = response_text.strip()
                if not response_text:
                    return "", "", 0.0

                result = EnhancedJSONParser.extract_json_robust(response_text)
                individual = result.get('individual_actors', '') or ""
                institutional = result.get('institutional_actors', '') or ""

                if 'error' not in result:
                    self.parsing_successes += 1
                else:
                    self.parsing_failures += 1

                input_tokens = response.usage.input_tokens if hasattr(response.usage, 'input_tokens') else 0
                output_tokens = response.usage.output_tokens if hasattr(response.usage, 'output_tokens') else 0
                estimated_cost = (input_tokens * PRICE_INPUT_PER_1M / 1_000_000) + (output_tokens * PRICE_OUTPUT_PER_1M / 1_000_000)
                self.total_cost += estimated_cost

                return individual, institutional, estimated_cost

            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] Exception: {type(e).__name__}: {str(e)}")
                self.parsing_failures += 1
                return "", "", 0.0
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list, list]:
        tasks = [
            self.analyze_single_article(row['title'], row['text'])
            for _, row in batch_df.iterrows()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        individuals, institutionals, costs = [], [], []
        for result in results:
            if isinstance(result, Exception):
                individuals.append("")
                institutionals.append("")
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                individuals.append(result[0])
                institutionals.append(result[1])
                costs.append(result[2])
        return individuals, institutionals, costs

    async def analyze_dataframe(self, df, output_filename="actor_analysis_results.csv"):
        output_path = os.path.join(self.output_dir, output_filename)
        all_individuals, all_institutionals, all_costs = [], [], []
        start_time = time.time()
        pbar = tqdm(total=len(df), desc="행위자 분석 중", unit="건")

        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i:min(i + self.batch_size, len(df))]
            try:
                individuals, institutionals, costs = await self.process_batch(batch_df)
                all_individuals.extend(individuals)
                all_institutionals.extend(institutionals)
                all_costs.extend(costs)
                pbar.update(len(batch_df))

                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    n = len(df)
                    temp_df = df.copy()
                    temp_df['개인행위자'] = all_individuals + [''] * (n - len(all_individuals))
                    temp_df['제도행위자'] = all_institutionals + [''] * (n - len(all_institutionals))
                    temp_df['비용'] = all_costs + [0.0] * (n - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
            except Exception as e:
                sz = len(batch_df)
                all_individuals.extend([""] * sz)
                all_institutionals.extend([""] * sz)
                all_costs.extend([0.0] * sz)
                self.parsing_failures += sz
                pbar.update(sz)

        pbar.close()
        result_df = df.copy()
        result_df['개인행위자'] = all_individuals
        result_df['제도행위자'] = all_institutionals
        result_df['비용'] = all_costs
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        total_count = len(result_df)
        indiv_count = (result_df['개인행위자'] != '').sum()
        inst_count  = (result_df['제도행위자'] != '').sum()
        print(f"\n{'='*60}")
        print(f"✅ 뉴스 행위자 분석 완료! (GPT-5.4-mini)")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 행위자 추출 현황:")
        print(f"   • 개인 행위자 포함 기사: {indiv_count}건 ({(indiv_count/total_count)*100:.1f}%)")
        print(f"   • 제도 행위자 포함 기사: {inst_count}건 ({(inst_count/total_count)*100:.1f}%)")
        print(f"\n📈 분석 성공률:")
        success_rate = (self.parsing_successes / total_count) * 100 if total_count > 0 else 0
        print(f"   • 성공: {self.parsing_successes}건 ({success_rate:.1f}%)")
        print(f"   • 실패: {self.parsing_failures}건 ({(self.parsing_failures/total_count)*100:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-5.4-mini"          # GPT-5.4-mini
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 사용 권장. 키를 코드에 직접 적은 채 커밋하지 마세요.
INPUT_CSV_PATH  = "Input CSV file path here"
OUTPUT_CSV_PATH = "Output CSV file path here"

# GPT-5.4-mini 전용 설정
# reasoning_effort options: none | low | medium | high | xhigh
# ※ extraction/classification 태스크에는 'none' 권장
REASONING_EFFORT = "none"
TEXT_VERBOSITY   = "medium"     # low | medium | high
# temperature: reasoning_effort='none'일 때만 유효 (None이면 비활성화)
TEMPERATURE      = None
# ※ GPT-4.1 버전은 temperature=0.0 고정이었습니다. 동일한 재현성이 필요하면 0.0으로 설정하세요(reasoning_effort="none"일 때만 유효).
DEBUG_MODE       = False

# 비용 추정용 단가 (USD / 1M tokens). OpenAI 공식 가격표 기준 gpt-5.4-mini 입력 $0.75, 출력 $4.50 (2026-11 확인). 가격이 바뀌면 수정하세요.
PRICE_INPUT_PER_1M  = 0.75
PRICE_OUTPUT_PER_1M = 4.50
# ──────────────────────────────────────────────────────────────────────

async def main():
    try:
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return

        output_dir  = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        analyzer = ActorAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir,
            reasoning_effort=REASONING_EFFORT,
            text_verbosity=TEXT_VERBOSITY,
            temperature=TEMPERATURE,
            debug=DEBUG_MODE
        )
        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")
        import traceback; traceback.print_exc()

def run_analysis():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    asyncio.run(main())

if __name__ == "__main__":
    run_analysis()